In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

In [2]:

import os
import pandas as pd
from langchain_groq.chat_models import ChatGroq
from dotenv import load_dotenv
from src.RAG.Constants.models import GroqModelList

gml = GroqModelList()

load_dotenv('../../Secrets/RAG.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY','')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY','')
print(len(GROQ_API_KEY))

/home/who/Documents/Coding/Freelance_Projects/Freelance_P-01_Cuisine-Menu-Recommendation-App/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


56


In [3]:
llm_groq = ChatGroq(
    model=gml.openai.gpt_oss_20b,
    temperature=0.7,
    api_key=GROQ_API_KEY,    
)

llm_groq.invoke(input='What LLM model are you?')

AIMessage(content='I’m built on OpenAI’s GPT‑4 architecture.', additional_kwargs={'reasoning_content': 'We need to answer: "What LLM model are you?" The user likely wants to know the underlying model. According to policy: The assistant is based on GPT-4 architecture. So we can say that I\'m based on GPT-4. We should be concise.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 78, 'total_tokens': 154, 'completion_time': 0.090508584, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.004729256, 'prompt_tokens_details': None, 'queue_time': 0.049077764, 'total_time': 0.09523784}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c0f73-c1df-7df2-9096-cd1af0895c75-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 76, 'total_tokens': 154, 'output_token_details'

In [4]:
graph_schema = """
node_label {node_property: dtype}:
Country {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
State {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
City {ids: str, name: str, coords: list[float], boundingbox: list[float]}
Area {ids: str, name: str}
Locality {ids: str, name: str}
Restaurant {ids: int, name: str, city: str, area: str, locality: str, cuisines: List[str], rating: float | null, address: str, coords: List[float] | null, chain: bool, city_id: str}
Menu {name: str, category: str, description: str, types: Literal["VEG", "NONVEG", "EGG", "UNKNOWN"], cuisine: str}
MainCuisine {name: str}

link_label {link_property: dtype}:
HAS_STATE {}
HAS_CITY {}
HAS_AREA {}
HAS_LOCALITY {}
HAS_RESTAURANT {}
HAS_MENU {price: int | null, rating: float | null}
SERVES_MAIN_CUISINE {}

Relationships:
(:Country)-[:HAS_STATE]->(:State)
(:State)-[:HAS_CITY]->(:City)
(:City)-[:HAS_AREA]->(:Area)
(:Area)-[:HAS_LOCALITY]->(:Locality)
(:Locality)-[:HAS_RESTAURANT]->(:Restaurant)
(:Restaurant)-[:HAS_MENU]->(:Menu)
(:Restaurant)-[:SERVES_MAIN_CUISINE]->(:MainCuisine)
"""

In [5]:
cypher_pattern = """
Allowed query patterns:

# Search for Restaurants based on location change to city.ids
PATTERN 1.1: Search for Restaurants by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.2: Search for Restaurants by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.3: Search for Restaurants by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

# Search for Menus based on location
PATTERN 2.1: Search for Menus by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m LIMIT 5000 DESC m.name

PATTERN 2.2: Search for Menus by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

PATTERN 2.3: Search for Menus by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

# Search for Menus based on cuisines



PATTERN_2: Restaurant by cuisine
MATCH (r:Restaurant)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_3: Restaurant by city AND cuisine
MATCH (r:Restaurant)-[:LOCATED_IN]->(c:City {name:$city}),
      (r)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_4: Similar restaurants
MATCH (r:Restaurant {name:$name})-[s:SIMILAR_TO]->(other)
RETURN other ORDER BY s.score DESC

Task:
- Select the SINGLE best pattern
- Output ONLY:
  pattern_name
  parameter_values

"""



In [6]:

system_prompt = f"""
You are a graph query planner.

You MUST follow these rules:
- Use ONLY the node labels listed below.
- Use ONLY the relationship types listed below.
- Use ONLY the properties listed below.
- Do NOT invent new labels, relationships, or properties.
- If a question cannot be answered using this schema, respond with:
  "NOT_ANSWERABLE_WITH_SCHEMA"

Graph schema:
{graph_schema}

You are NOT allowed to use any other schema elements.

"""

In [3]:
from src.ETL.Config.graph_pool import GraphPool

graph = GraphPool.get_graph(graph_name='test')
graph

In [4]:
# get best rated menus from best rated restaurants serving thai cuisines in Koramangala
from time import time


graph_query = """
MATCH (:Area {name: 'Koramangala'})-[*2]->(rstn:Restaurant)-[]->(:MainCuisine {name: 'Thai'})
WHERE rstn.rating > 4.0
MATCH (rstn)-[link:HAS_MENU]-(menu:Menu)
WHERE link.rating > 4.0
RETURN DISTINCT
    rstn.name,
    menu.name,
    menu.types,
    link.price,
    link.rating
ORDER BY link.rating DESC
LIMIT 2000
"""
st = time()
result = graph.query(graph_query).result_set
print(f"Time taken for graph query: {(time() - st)*1000:.2f} ms")
df = pd.DataFrame([row for row in result], columns=['Restautant','Name', 'Type', 'Price', 'Rating'])


df.head(10)

Time taken for graph query: 479.49 ms


,Restautant,Name,Type,Price,Rating
0,Bamey's Restro Cafe,Paneer Chilly,VEG,315.0,5.0
1,Momo Rice Noodles,Fish Manchurian Gravy,NONVEG,370.0,5.0
2,Momo Rice Noodles,Veg Chilli Garlic Curry with Rice,VEG,250.0,5.0
3,Momo Rice Noodles,Chilli Crispy Fish Dry,NONVEG,370.0,5.0
4,Momo Rice Noodles,Spicy Curd Chicken Lollipop ( 6 pic),NONVEG,331.0,5.0
5,Momo Rice Noodles,Veg Butter Garlic Noodles,VEG,225.0,5.0
6,Beijing Bites,Hunan Tofu,VEG,299.0,5.0
7,Momo Rice Noodles,Pork Schezwan Fried Rice,NONVEG,290.0,5.0
8,Momo Rice Noodles,Seafood Mushroom Clear Soup,NONVEG,195.0,5.0
9,Thai Basil,Stir Fried Chicken Red Curry Paste,NONVEG,480.0,5.0


In [5]:
from typing import Literal, Dict, Any, List, Set


In [6]:
def get_competitors_data(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
) -> Dict[str, Any] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {ids: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    MATCH (r)-[link:HAS_MENU]->(menu:Menu)
    WHERE r.rating IS NOT NULL AND r.rating >= $min_rating
    RETURN  
        r.name, r.area, r.cuisines, r.rating, r.chain,
        min(link.rating) AS min_menu_rating,
        avg(link.rating) AS avg_menu_rating,
        max(link.rating) AS max_menu_rating,
        min(link.price) AS min_menu_price,
        avg(link.price) AS avg_menu_price,
        max(link.price) AS max_menu_price,
        stDev(link.price) AS sd_menu_price
    ORDER BY r.rating DESC, r.name ASC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    columns = ['name', 'area', 'cuisines', 'rating', 'chain',"min_menu_rating", 'avg_menu_rating',"max_menu_rating", 'min_menu_price','avg_menu_price','max_menu_price','sd_menu_price']
    rqrd_data = {
        key: [
            ', '.join(str(item) for item in data[columns.index(key)]) 
            if isinstance(data[columns.index(key)], list) 
            else data[columns.index(key)]
            for data in result
        ]
        for key in columns
    }
    return pd.DataFrame(rqrd_data) if output=='dataframe' else rqrd_data

In [7]:
from src.RAG.Config.tool_models import GetCompetitorDataModels

f_params = GetCompetitorDataModels.FunctionParams(
    q_params=GetCompetitorDataModels.QueryParams(
        area='area_Koramangala__city_Bangalore-relation:7902476',#'Indiranagar',
        cuisine='Thai',
        limit=200,
    ),
    output='dataframe',
)
cmpt_data = get_competitors_data(**f_params.model_dump())
cmpt_data

,name,area,cuisines,rating,chain,min_menu_rating,avg_menu_rating,max_menu_rating,min_menu_price,avg_menu_price,max_menu_price,sd_menu_price
0,Yuki,Koramangala,"Thai, Pan-Asian",4.6,False,3.5,4.609877,5.0,99,427.093333,795,109.136904
1,Phra Nakhon - Thai Cafe,Koramangala,Thai,4.5,False,2.0,4.346226,5.0,90,403.451852,514,60.325476
2,Thai Basil,Koramangala,Thai,4.5,False,2.9,4.477895,5.0,150,542.857143,840,95.591667
3,Beijing Bites,Koramangala,"Chinese, Thai",4.4,True,2.0,4.362857,5.0,60,285.735915,499,60.862853
4,Love Momo,Koramangala,"Chinese, Thai",4.4,False,1.5,3.509091,4.4,139,233.210526,299,35.804398
5,Momo Rice Noodles,Koramangala,"Chinese, Thai",4.4,False,1.5,4.204727,5.0,45,275.208861,580,86.046724
6,Bamey's Restro Cafe,Koramangala,"Chinese, Thai",4.3,False,1.5,4.102000,5.0,155,323.788018,525,77.477125
7,Samruddhi Dhaba,Koramangala,"Thai, North Indian",4.3,False,3.8,3.800000,3.8,15,146.486486,235,48.732046
8,Hornbill Naga Kitchen,Koramangala,"Chinese, Thai",4.0,False,3.6,4.344444,5.0,40,238.428571,470,105.709466
9,Shimray Restaurant,Koramangala,"Continental, Thai",4.0,False,2.3,3.614286,4.5,49,184.567568,489,102.195222


In [12]:
def get_competitors_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: Set[str]={"ids", "city_id"},
)-> Dict[str, Any] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(rstn:Restaurant)
    MATCH (rstn)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    MATCH (rstn)-[link:HAS_MENU]->(menu:Menu)
    WHERE link.rating IS NOT NULL AND link.rating >= $min_menu_rating

    WITH rstn, menu, link
    ORDER BY rstn.name ASC, link.rating DESC, menu.name ASC

    WITH rstn, collect({
        rstn: properties(rstn),
        menu: properties(menu),
        food: properties(link)
    })[0..20] AS top_menus

    UNWIND top_menus AS merged
    RETURN merged
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    full_data = {
        f"{key1}_{key2}".replace("menu_", "food_", 1): [
            (', '.join(str(item) for item in x) if isinstance(x, list) else x)
            if (x:= items[0][key1].get(key2, ''))
            else False
            for items in result
        ]
        for key1 in result[0][0].keys()
        for key2 in result[0][0][key1].keys()
        if f"{key1}_{key2}" not in exclude
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data


In [13]:
from src.RAG.Config.tool_models import GetCompetitorMenuModels

f_params = GetCompetitorMenuModels.FunctionParams(
    q_params=GetCompetitorMenuModels.QueryParams(
        area='Indiranagar',
        cuisine='Thai',
        min_menu_rating=4.0,
        limit=500,
    ),
    output='dataframe',
    exclude={'rstn_city_id','rstn_ids', "rstn_city", "rstn_locality", "rstn_address", "rstn_coords"}
)
data = get_competitors_menu(**f_params.model_dump())
data

,rstn_name,rstn_area,rstn_cuisines,rstn_rating,rstn_chain,food_name,food_types,food_price,food_rating
0,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Beijing Chilli Prawns,NONVEG,260,5.0
1,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Hot Chilli Baby corn,VEG,150,5.0
2,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Mixed Vegetable in Green Chilli Gravy,VEG,180,5.0
3,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Prawns Chinese Chop suey,NONVEG,220,5.0
4,China Garten,Indiranagar,"Chinese, Thai",4.3,False,Spicy Vegetable Curry,VEG,170,5.0
...,...,...,...,...,...,...,...,...,...
111,Thai Basil,Indiranagar,Thai,4.6,True,Slice Grilled Beef Salad,NONVEG,540,5.0
112,Thai Basil,Indiranagar,Thai,4.6,True,Stir-Fried Squid With Curry Powder,NONVEG,595,5.0
113,Thai Basil,Indiranagar,Thai,4.6,True,Tomkha Prawns,NONVEG,590,5.0
114,Thai Basil,Indiranagar,Thai,4.6,True,Tomkha Seafood,NONVEG,495,5.0


In [14]:
def get_menu_benchmark(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
)-> Dict[str, Any] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE toLower(m.name) CONTAINS toLower($menu_name)
        AND link.price IS NOT NULL
    RETURN
        m.name,
        link.price,
        link.rating,
        r.name,
        r.rating
    ORDER BY r.name ASC, link.price DESC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    columns = ['food_name', 'food_price', 'food_rating', 'rstn_name', 'rstn_rating']
    full_data = {
        key: [
            item[columns.index(key)]
            for item in result
        ]
        for key in columns
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data
    

In [15]:
from src.RAG.Config.tool_models import GetMenuBenchmarkModels

f_params = GetMenuBenchmarkModels.FunctionParams(
    q_params=GetMenuBenchmarkModels.QueryParams(
        area='Indiranagar',
        cuisine='South Indian',
        menu_name='Masala Dosa',
        limit=250
    ),
    output='dataframe'
)
data = get_menu_benchmark(**f_params.model_dump())
data

ResponseError: Query timed out

In [ ]:
def get_menu_oppourtunities(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
)-> Dict[str, Any] | pd.DataFrame:
    graph_query= """
    MATCH (:Area {name:$area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name:$cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE link.rating IS NOT NULL AND link.rating >= $min_menu_rating
    WITH
        m.name AS menu_name,
        m.types AS types,
        count(DISTINCT r.ids) AS competitor_count,
        avg(link.rating) AS avg_menu_rating,
        min(link.price) AS min_menu_price,
        avg(link.price) AS avg_menu_price,
        max(link.price) AS max_menu_price,
        stDev(link.price) AS sd_menu_price
    RETURN
        menu_name, types,
        competitor_count,
        avg_menu_rating,
        min_menu_price,
        avg_menu_price,
        max_menu_price,
        sd_menu_price
    ORDER BY competitor_count ASC, avg_menu_rating DESC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    columns = ['food_name', 'food_types', 'no_of_competitors', 'avg_menu_rating', "min_menu_price", "avg_menu_price", "max_menu_price", "sd_menu_price"]
    full_data = {
        key: [
            item[columns.index(key)]
            for item in result
        ]
        for key in columns
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data

In [ ]:
from src.RAG.Config.tool_models import GetMenuOpportunitiesModels

f_params = GetMenuOpportunitiesModels.FunctionParams(
    q_params=GetMenuOpportunitiesModels.QueryParams(
        area='Koramangala',
        cuisine='Thai',
        min_menu_rating=4.0,
        limit=250,
    ),
    output='dataframe'
)

data = get_menu_oppourtunities(**f_params.model_dump())
data

,food_name,food_types,no_of_competitors,avg_menu_rating,min_menu_price,avg_menu_price,max_menu_price,sd_menu_price
0,Yum Woon Sen Talay,NONVEG,1,5.0,494,494.0,494,0.0
1,Dragan Fish Dry,NONVEG,1,5.0,370,370.0,370,0.0
2,Lab Kai,NONVEG,1,5.0,378,378.0,378,0.0
3,Mixed Steamed Rice - Koithio,NONVEG,1,5.0,319,319.0,319,0.0
4,Shredded Chicken In Chilli Oil.,NONVEG,1,5.0,445,445.0,445,0.0
...,...,...,...,...,...,...,...,...
245,Pad Prieow Wan Tao Hoo,VEG,1,4.8,390,390.0,390,0.0
246,Momo Platter Buff 8 Pcs,NONVEG,1,4.8,350,350.0,350,0.0
247,Veg Chilli Garlic Rice Bowl,VEG,1,4.8,269,269.0,269,0.0
248,Honey Spicy Pork Dry,NONVEG,1,4.8,580,580.0,580,0.0


In [ ]:
def get_overpriced_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
)-> Dict[str, Any] | pd.DataFrame:
    graph_query= """
    MATCH (:Area {name:$area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name:$cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE link.price IS NOT NULL AND link.rating IS NOT NULL
    WITH
        m.name AS food_name,
        min(link.price) AS min_food_price,
        avg(link.price) AS avg_food_price,
        max(link.price) AS max_food_price,
        stDev(link.price) AS sd_food_price,
        min(link.rating) AS min_food_rating,
        avg(link.rating) AS avg_food_rating,
        max(link.rating) AS max_food_rating,
        count(*) AS listings
    WHERE listings >= $min_listings AND avg_food_rating <= $max_avg_rating
    RETURN food_name, min_food_price, avg_food_price, max_food_price, sd_food_price, 
    min_food_rating, avg_food_rating, max_food_rating, listings
    ORDER BY food_name ASC, avg_food_rating DESC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    columns = ['food_name', 'min_food_price','avg_food_price', 'max_food_price', 'sd_food_price', 'min_food_rating','avg_food_rating', 'max_food_rating','listings']
    full_data = {
        key: [
            item[columns.index(key)]
            for item in result
        ]
        for key in columns
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data

In [ ]:
from src.RAG.Config.tool_models import GetOverpricedMenuModels

f_params = GetOverpricedMenuModels.FunctionParams(
    q_params=GetOverpricedMenuModels.QueryParams(
        area='Koramangala',
        cuisine='North Indian',
        min_listings=5,
        max_avg_rating=4.0,
        limit=500,
    ),
    output='dataframe'
)

data = get_overpriced_menu(**f_params.model_dump())
data

,food_name,min_food_price,avg_food_price,max_food_price,sd_food_price,min_food_rating,avg_food_rating,max_food_rating,listings
0,Aloo Paratha,45,108.882353,160,33.460578,1.5,3.835294,5.0,17
1,Butter Chicken,269,363.000000,419,60.726161,2.6,3.642857,5.0,7
2,Butter Chicken Rice Bowl,220,310.833333,399,59.834494,2.5,3.650000,4.4,6
3,Chicken Biryani,249,343.428571,390,49.772339,2.8,3.771429,4.2,7
4,Chicken Do Pyaza,199,239.000000,339,61.644140,2.7,3.900000,4.8,5
5,Chicken Fried Rice,100,233.923077,339,66.501205,2.4,3.846154,4.7,13
6,Chicken Lollipop,150,283.200000,359,78.506051,2.6,3.640000,4.6,5
7,Chicken Manchurian,150,257.200000,319,71.527617,2.8,3.760000,5.0,5
8,Chicken Noodles,169,233.571429,319,49.523924,1.7,3.728571,4.9,7
9,Dal Tadka,110,219.090909,395,73.451283,2.8,3.990909,4.9,11


In [ ]:
def get_premium_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
)-> Dict[str, Any] | pd.DataFrame:
    graph_query= """
    MATCH (:Area {name:$area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name:$cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE link.price IS NOT NULL AND link.rating IS NOT NULL
    WITH
        m.name AS food_name,
        min(link.price) AS min_food_price,
        avg(link.price) AS avg_food_price,
        max(link.price) AS max_food_price,
        stDev(link.price) AS sd_food_price,
        min(link.rating) AS min_food_rating,
        avg(link.rating) AS avg_food_rating,
        max(link.rating) AS max_food_rating,
        count(*) AS listings
    WHERE listings >= $min_listings AND avg_food_rating >= $min_avg_rating
    RETURN food_name, min_food_price, avg_food_price, max_food_price, sd_food_price, 
    min_food_rating, avg_food_rating, max_food_rating, listings
    ORDER BY food_name ASC, avg_food_rating DESC, avg_food_price DESC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    columns = ['food_name', 'min_food_price','avg_food_price', 'max_food_price', 'sd_food_price', 'min_food_rating', 'avg_food_rating','max_food_rating', 'listings']
    full_data = {
        key: [
            item[columns.index(key)]
            for item in result
        ]
        for key in columns
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data

In [ ]:
from src.RAG.Config.tool_models import GetPremiumMenuModels

f_params = GetPremiumMenuModels.FunctionParams(
    q_params=GetPremiumMenuModels.QueryParams(
        area='Koramangala',
        cuisine='North Indian',
        min_listings=2,
        min_avg_rating=4.5,
        limit=500,
    ),
    output='dataframe'
)

data = get_premium_menu(**f_params.model_dump())
data

,food_name,min_food_price,avg_food_price,max_food_price,sd_food_price,min_food_rating,avg_food_rating,max_food_rating,listings
0,5 Pc Roti + Aloo Jerra + Dal,199,199.0,199,0.000000,4.7,4.733333,4.8,3
1,5 Pc Roti + Chicken Masala + Dal,265,265.0,265,0.000000,4.7,4.850000,5.0,2
2,5 Pc Roti + Chole Masala + Dal,199,199.0,199,0.000000,4.3,4.600000,4.9,3
3,5 Pc Roti + Paneer Masala + Dal,199,199.0,199,0.000000,4.3,4.666667,5.0,3
4,5 Pc Roti+ Bhindi Masala + Dal,249,249.0,249,0.000000,4.3,4.500000,4.7,3
...,...,...,...,...,...,...,...,...,...
153,Veg Hot & Sour Soup,129,134.0,139,7.071068,4.2,4.600000,5.0,2
154,Veg Kolapuri,189,196.5,199,5.000000,3.9,4.600000,5.0,4
155,Veg Tadka,185,185.0,185,0.000000,4.8,4.900000,5.0,3
156,Vegetable Pulao,175,285.0,395,155.563492,4.5,4.700000,4.9,2


In [ ]:
def get_specific_competitor_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
)-> Dict[str, Any] | pd.DataFrame:
    graph_query= """
    MATCH (rstn:Restaurant {ids: $rstn_id})-[link:HAS_MENU]->(m:Menu)
    RETURN
        rstn.name,
        rstn.area,
        m.name,
        m.types,
        link.price,
        link.rating
    ORDER BY link.rating DESC, m.name ASC
    LIMIT $limit

    """
    result = graph.query(graph_query, q_params).result_set
    full_data = result
    columns = ['rstn_name', 'rstn_area', 'food_name', 'food_types','food_price', 'food_rating']
    full_data = {
        key: [
            item[columns.index(key)]
            for item in result
        ]
        for key in columns
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data

In [ ]:
from src.RAG.Config.tool_models import GetSpecificCompetitorMenuModels

f_params = GetSpecificCompetitorMenuModels.FunctionParams(
    q_params=GetSpecificCompetitorMenuModels.QueryParams(
        rstn_id=418,
        limit=200,
    ),
    output='dataframe',
)

data = get_specific_competitor_menu(**f_params.model_dump())
data

,rstn_name,rstn_area,food_name,food_types,food_price,food_rating
0,Ande Ka Funda,Kadubeesanahalli,Boneless Chicken Meal,NONVEG,271.0,NaN
1,Ande Ka Funda,Kadubeesanahalli,Mix Veg Masala,VEG,NaN,NaN
2,Ande Ka Funda,Kadubeesanahalli,Paneer Mushroom,VEG,NaN,NaN
3,Ande Ka Funda,Kadubeesanahalli,Poori Sabjis,VEG,99.0,4.9
4,Ande Ka Funda,Kadubeesanahalli,Basmati Jeera Rice,VEG,99.0,4.8
...,...,...,...,...,...,...
130,Ande Ka Funda,Kadubeesanahalli,Special Veg Meal,VEG,159.0,2.8
131,Ande Ka Funda,Kadubeesanahalli,Pepsi [250 Ml],VEG,30.0,2.7
132,Ande Ka Funda,Kadubeesanahalli,Aloo Jeera,VEG,NaN,2.6
133,Ande Ka Funda,Kadubeesanahalli,Roti Choka Meal,VEG,100.0,2.3


In [ ]:
def recommend_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
)-> Dict[str, Any] | pd.DataFrame:
    graph_query= """
    MATCH (:Area {name:$area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name:$cuisine})
    MATCH (r)-[link:HAS_MENU]->(m:Menu)
    WHERE link.rating IS NOT NULL AND link.rating >= $min_menu_rating
    WITH
        m.name AS menu_name,
        m.types AS types,
        avg(link.price) AS avg_price,
        avg(link.rating) AS avg_rating
    RETURN menu_name, types, avg_price, avg_rating
    ORDER BY avg_rating DESC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    full_data = result
    columns = ['food_name', 'food_types', 'avg_food_price', 'avg_food_rating']
    full_data = {
        key: [
            item[columns.index(key)]
            for item in result
        ]
        for key in columns
    }
    return pd.DataFrame(full_data) if output=='dataframe' else full_data

In [ ]:
from src.RAG.Config.tool_models import RecommendMenuModels

f_params = RecommendMenuModels.FunctionParams(
    q_params=RecommendMenuModels.QueryParams(
        area='Indiranagar',
        cuisine='Continental',
        min_menu_rating=4.0,
        limit=300,
    ),
    output='dataframe',
)

data = recommend_menu(**f_params.model_dump())
data

,food_name,food_types,avg_food_price,avg_food_rating
0,Banana Split Smoothie,VEG,260.0,5.0
1,Green G Special Fruit Bowl,VEG,170.0,5.0
2,Peas Pulao,VEG,285.0,5.0
3,Spicy Cauliflower Bites,VEG,250.0,5.0
4,Primavera Panini,VEG,249.0,5.0
...,...,...,...,...
295,Abc Juice,VEG,220.0,4.8
296,Truffle Mushroom Omelette With Spinach,NONVEG,310.0,4.8
297,Smoke House Fiery Bbq Chicken Wings,NONVEG,480.0,4.8
298,Neer Dosa (Two),VEG,95.0,4.8


In [ ]:
from langgraph.graph import START, StateGraph, END

